# 02b_Metadaten_Features_Modellierung

Deterministische LogReg + Permutation Importance; finaler Export nach `../features/metadata_features.csv`.

## 1) Setup & Laden

In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, json, re, warnings
warnings.filterwarnings("ignore")

NOTEBOOKS_DIR = Path(".").resolve()
FEATURES_DIR = Path("..").resolve() / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

def read_parquet_or_csv(base_name: str):
    p_parquet = Path(f"{base_name}.parquet")
    p_csv = Path(f"{base_name}.csv")
    if p_parquet.exists():
        df = pd.read_parquet(p_parquet); print("Geladen (Parquet):", p_parquet.resolve()); return df
    elif p_csv.exists():
        df = pd.read_csv(p_csv); print("Geladen (CSV):", p_csv.resolve()); return df
    else:
        raise FileNotFoundError(f"Weder {p_parquet} noch {p_csv} gefunden. Bitte 02a zuerst ausführen.")

feat_raw = read_parquet_or_csv("metadata_features_raw")
print("Shape:", feat_raw.shape)
display(feat_raw.head(5))

label_col = "is_viral" if "is_viral" in feat_raw.columns else ("is_viral_proxy" if "is_viral_proxy" in feat_raw.columns else None)
assert label_col is not None, "Kein Label gefunden."
feat_raw = feat_raw[feat_raw[label_col].notna()].reset_index(drop=True)


## 2) Preprocessing & Split

In [ ]:

import numpy as np, pandas as pd, re

df = feat_raw.copy()
X_df = df[[c for c in df.columns if re.match(r"^(acct|txt|len|time)_", c)]]
y = df[label_col].astype(int).to_numpy()

cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object" or str(X_df[c].dtype) == "category"]
X_df = pd.get_dummies(X_df, columns=cat_cols, dummy_na=True)

for c in X_df.columns:
    X_df[c] = pd.to_numeric(X_df[c], errors="coerce").fillna(0.0)

feature_names = X_df.columns.tolist()
X = X_df.to_numpy().astype(float)

pos_idx = np.where(y==1)[0]; neg_idx = np.where(y==0)[0]
rng.shuffle(pos_idx); rng.shuffle(neg_idx)
pos_split = int(len(pos_idx)*0.8)
neg_split = int(len(neg_idx)*0.8)
train_idx = np.concatenate([pos_idx[:pos_split], neg_idx[:neg_split]])
valid_idx = np.concatenate([pos_idx[pos_split:], neg_idx[neg_split:]])

Xtr, Xva = X[train_idx], X[valid_idx]
ytr, yva = y[train_idx], y[valid_idx]

mu = Xtr.mean(axis=0); sigma = Xtr.std(axis=0); sigma[sigma==0]=1.0
Xtr_z = (Xtr - mu)/sigma; Xva_z = (Xva - mu)/sigma


## 3) Logistische Regression (NumPy, L2)

In [ ]:

import numpy as np

def sigmoid(z):
    z = np.clip(z, -20, 20)
    return 1.0/(1.0 + np.exp(-z))

def fit_logreg(X, y, l2=1.0, lr=0.1, steps=1500):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for t in range(steps):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y))/n + (l2*w)/n
        grad_b = np.mean(p - y)
        w -= lr*grad_w; b -= lr*grad_b
    return w, b

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def roc_auc_score(y_true, y_score):
    order = np.argsort(y_score)
    y = y_true[order]
    n_pos = y.sum(); n_neg = len(y)-n_pos
    if n_pos==0 or n_neg==0: return np.nan
    rank = np.arange(1, len(y)+1)
    s = rank[y==1].sum()
    return float((s - n_pos*(n_pos+1)/2) / (n_pos*n_neg))

w, b = fit_logreg(Xtr_z, ytr, l2=1.0, lr=0.1, steps=1500)
p_tr = predict_proba(Xtr_z, w, b); p_va = predict_proba(Xva_z, w, b)

def bin_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    prec = tp / max(1, tp+fp); rec = tp / max(1, tp+fn)
    f1 = 2*prec*rec / max(1e-9, (prec+rec)); acc = (tp+tn)/len(y_true)
    auc = roc_auc_score(y_true, y_prob)
    return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1,"auc":auc}

report = {"train": bin_metrics(ytr, p_tr), "valid": bin_metrics(yva, p_va),
          "params":{"l2":1.0,"lr":0.1,"steps":1500,"seed":42},
          "n_features": int(Xtr_z.shape[1])}
report


## 4) Permutation Importance

In [ ]:

base_auc = report["valid"]["auc"]
imp = []
for j, name in enumerate(feature_names):
    Xva_perm = Xva_z.copy()
    idx = np.arange(len(Xva_perm)); rng.shuffle(idx)
    Xva_perm[:, j] = Xva_perm[idx, j]
    p_perm = predict_proba(Xva_perm, w, b)
    auc_perm = roc_auc_score(yva, p_perm)
    delta = base_auc - auc_perm
    imp.append((name, float(delta)))

imp_df = pd.DataFrame(sorted(imp, key=lambda x: x[1], reverse=True), columns=["feature","perm_importance_auc_drop"])
imp_path = NOTEBOOKS_DIR / "metadata_feature_importance.csv"
imp_df.to_csv(imp_path, index=False)
print("Feature-Importance gespeichert:", imp_path.resolve())
display(imp_df.head(15))


## 5) Export finaler Features

In [ ]:

selected = imp_df[imp_df["perm_importance_auc_drop"] > 0.0]["feature"].tolist()
KEY_COLS = [c for c in ["video_id","group"] if c in feat_raw.columns]
LABEL_COLS = [c for c in ["is_viral","is_viral_proxy"] if c in feat_raw.columns]
available = [c for c in selected if c in feat_raw.columns]
final_df = feat_raw[KEY_COLS + LABEL_COLS + available].copy()

out_path = FEATURES_DIR / "metadata_features.csv"
final_df.to_csv(out_path, index=False)
print("Finale Feature-Datei gespeichert:", out_path.resolve())
display(final_df.head(5))


## 6) Report speichern

In [ ]:

report_path = NOTEBOOKS_DIR / "metadata_model_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Report gespeichert:", report_path.resolve())
print(json.dumps(report, indent=2, ensure_ascii=False))


---

**Übergabe an T4**: `features/metadata_features.csv` mergen mit Audio/Video; Training & Evaluation übernehmen.